In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

# ==========================================
# 1. 配置与超参数
# ==========================================
d = 20                   # 维度
K_phi = 1                # 对抗网络迭代次数
K_u = 2                  # 原问题网络迭代次数
tau_eta = 0.04           # 对抗网络学习率
tau_theta = 0.015        # 原问题网络学习率
Nr = 4000 * d            # 区域内采样点数
Nb = 40 * d * d          # 边界采样点数
alpha = 20000 * Nb       # 边界权重
hls_theta = 20           # 原问题网络隐藏层大小
hls_eta = 50             # 对抗网络隐藏层大小
NUM_ITERS = 2000         # 迭代次数 (为演示方便设为2000，原代码为20000)

# 检测GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 2. PDE 系数与真解定义
# ==========================================
# 注意：PyTorch中输入x形状为 (Batch, Dim)

def a0(x):
    # sum(x^2)
    return 1.0 + torch.sum(x ** 2, dim=1, keepdim=True)

def rho0(x):
    # x[:, 0] 是第一维, x[:, 1] 是第二维
    return (np.pi * (x[:, 0:1] ** 2) + (x[:, 1:2] ** 2)) / 2.0

def rho1(x):
    return ((np.pi ** 2) * (x[:, 0:1] ** 2) + (x[:, 1:2] ** 2)) / 4.0

# 这是一个复用的计算项，用于构造 PDE 的右端项 f
def f_func_full(x, _a, _rho0, _rho1, _cos):
    return 4.0 * _rho1 * _a * torch.sin(_rho0) + \
           (2.0 * _rho1 - 4.0 * _rho0 - (np.pi + 1) * _a) * _cos

def f_func(x):
    _a = a0(x)
    _r0 = rho0(x)
    _r1 = rho1(x)
    return f_func_full(x, _a, _r0, _r1, torch.cos(_r0))

def u_true(x):
    # 真解
    return torch.sin((np.pi * x[:, 0:1] ** 2 + x[:, 1:2] ** 2) / 2.0)

def g_func(x):
    return u_true(x)

# ==========================================
# 3. 神经网络定义
# ==========================================

# 原问题网络 u_theta
class PrimalNet(nn.Module):
    def __init__(self, input_dim, hidden_size):
        super(PrimalNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_size), nn.Tanh(),
            nn.Linear(hidden_size, hidden_size), nn.Tanh(),
            nn.Linear(hidden_size, hidden_size), nn.Softplus(),
            nn.Linear(hidden_size, hidden_size), nn.Tanh(),
            nn.Linear(hidden_size, hidden_size), nn.Softplus(),
            nn.Linear(hidden_size, hidden_size), nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x):
        return self.net(x)

# 自定义 Sinc 激活函数 (sin(x)/x)
class Sinc(nn.Module):
    def __init__(self):
        super(Sinc, self).__init__()
        self.epsilon = 1e-7

    def forward(self, x):
        return torch.sin(x) / (x + self.epsilon)

# 对抗网络 phi_eta
class AdversarialNet(nn.Module):
    def __init__(self, input_dim, hidden_size):
        super(AdversarialNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_size), nn.Tanh(),
            nn.Linear(hidden_size, hidden_size), nn.Tanh(),
            nn.Linear(hidden_size, hidden_size), nn.Softplus(),
            nn.Linear(hidden_size, hidden_size), Sinc(),
            nn.Linear(hidden_size, hidden_size), nn.Softplus(),
            nn.Linear(hidden_size, hidden_size), Sinc(),
            nn.Linear(hidden_size, hidden_size), Sinc(),
            nn.Linear(hidden_size, hidden_size), nn.Softplus(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x):
        return self.net(x)

# 初始化模型
u_theta = PrimalNet(d, hls_theta).to(device)
phi_eta = AdversarialNet(d, hls_eta).to(device)

class RFMAdeversarialNet(nn.Module):
    def __

# 优化器 (使用 Adagrad 以匹配 Julia 代码)
opt_theta = optim.Adagrad(u_theta.parameters(), lr=tau_theta)
opt_eta = optim.Adagrad(phi_eta.parameters(), lr=tau_eta)

# ==========================================
# 4. 损失函数与梯度计算
# ==========================================

def get_grad(net, x):
    """计算网络输出相对于输入 x 的梯度"""
    x.requires_grad_(True)
    y = net(x)
    # create_graph=True 是必须的，因为我们要在损失函数中使用这个梯度，
    # 并且后续还需要对参数求导（即需要梯度的梯度）
    grads = torch.autograd.grad(
        outputs=y, 
        inputs=x, 
        grad_outputs=torch.ones_like(y), 
        create_graph=True, 
        retain_graph=True,
        only_inputs=True
    )[0]
    return grads

def I_func(x, _grad_u, _phi, _a):
    # 弱形式的核心被积函数
    # term1: sum(grad_u * grad_phi) * a
    grad_phi = get_grad(phi_eta, x)
    t1 = torch.sum(_grad_u * grad_phi, dim=1, keepdim=True) * _a
    
    # term2: sum(grad_u^2) / 2
    t2 = torch.sum(_grad_u * _grad_u, dim=1, keepdim=True) / 2.0
    
    # Result
    return (t2 - f_func(x)) * _phi - t1

def loss_int(xr):
    # 区域积分损失
    _phi = phi_eta(xr)
    _grad_u = get_grad(u_theta, xr)
    _a = a0(xr)
    
    integrand = I_func(xr, _grad_u, _phi, _a)
    t1 = torch.sum(integrand)
    
    # log( (sum(I))^2 / sum(phi^2) )
    numerator = t1 * t1
    denominator = torch.sum(_phi * _phi)
    
    # 加上一个小量防止除零
    return torch.log(numerator / (denominator + 1e-8))

def loss_bndry(xb):
    # 边界损失
    k = u_theta(xb) - g_func(xb)
    return torch.sum(k * k) / Nb

# ==========================================
# 5. 训练循环
# ==========================================

def train_step():
    # 1. 区域内采样 [-1, 1]
    xr = (2.0 * torch.rand(Nr, d) - 1.0).to(device)
    xr.requires_grad_(True)
    
    # 2. 边界采样
    # 创建随机数据
    xb = (2.0 * torch.rand(Nb, d) - 1.0)
    # 随机选择维度将其置为 -1 或 1
    # PyTorch 生成掩码
    for i in range(Nb):
        j = np.random.randint(0, d) # 0 到 d-1
        val = 1.0 if np.random.random() > 0.5 else -1.0
        xb[i, j] = val
        
    xb = xb.to(device)
    
    # --- 更新原问题网络 u_theta (Primal) ---
    for _ in range(K_u):
        opt_theta.zero_grad()
        l_int = loss_int(xr)
        l_bnd = loss_bndry(xb)
        
        # 总损失 = 内部对抗损失 + 边界惩罚
        total_loss = l_int + alpha * Nb * l_bnd
        
        total_loss.backward()
        opt_theta.step()

    # --- 更新对抗网络 phi_eta (Adversary) ---
    for _ in range(K_phi):
        opt_eta.zero_grad()
        # 对抗网络试图最大化积分比率，即最小化负的损失
        # Julia代码: ADAGrad(-lr)，这里我们在Loss前加负号
        l_adv = -loss_int(xr)
        
        l_adv.backward()
        opt_eta.step()
        
    return l_int.item(), l_bnd.item()

print("Starting training...")
start_time = time.time()

loss_history = []

try:
    for i in range(1, NUM_ITERS + 1):
        l_i, l_b = train_step()
        
        if i % 100 == 0:
            print(f"Iter {i}/{NUM_ITERS} | Int Loss: {l_i:.4f} | Bndry Loss: {l_b:.6f}")
            loss_history.append((l_i, l_b))

except KeyboardInterrupt:
    print("Training interrupted.")

print(f"Training done in {time.time() - start_time:.2f} seconds.")

# ==========================================
# 6. 可视化 (Plots)
# ==========================================
# 切换到 CPU 进行绘图
u_theta.cpu()
phi_eta.cpu()

# 创建网格
grid_size = 50
plot_range = np.linspace(-1, 1, grid_size)
X, Y = np.meshgrid(plot_range, plot_range)

# 构造输入: 前两维变化，其余维为0
input_grid = np.zeros((grid_size * grid_size, d), dtype=np.float32)
flat_X = X.flatten()
flat_Y = Y.flatten()
input_grid[:, 0] = flat_X
input_grid[:, 1] = flat_Y

input_tensor = torch.from_numpy(input_grid)

# 预测与真解
with torch.no_grad():
    u_pred = u_theta(input_tensor).numpy().reshape(grid_size, grid_size)
    u_exact = u_true(input_tensor).numpy().reshape(grid_size, grid_size)
    error = np.abs(u_pred - u_exact)

# 绘图
fig = plt.figure(figsize=(18, 5))

# Plot 1: True Solution
ax1 = fig.add_subplot(1, 3, 1, projection='3d')
ax1.plot_surface(X, Y, u_exact, cmap='viridis')
ax1.set_title("True Solution u(x)")

# Plot 2: Neural Network Approximation
ax2 = fig.add_subplot(1, 3, 2, projection='3d')
ax2.plot_surface(X, Y, u_pred, cmap='plasma')
ax2.set_title("Approximation u_theta(x)")

# Plot 3: Error
ax3 = fig.add_subplot(1, 3, 3)
im = ax3.contourf(X, Y, error, levels=20, cmap='inferno')
fig.colorbar(im, ax=ax3)
ax3.set_title("Absolute Error |u - u_theta|")

plt.tight_layout()
plt.show()

# 保存模型 (类似 @save)
torch.save(u_theta.state_dict(), "primal.pth")
torch.save(phi_eta.state_dict(), "adversary.pth")
print("Models saved.")


Using device: cuda
Starting training...


C:\ProgramData\anaconda3\Lib\site-packages\torch\autograd\graph.py:823: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\cuda\CublasHandlePool.cpp:180.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Iter 100/2000 | Int Loss: 11.1640 | Bndry Loss: 0.012318
Iter 200/2000 | Int Loss: 10.8742 | Bndry Loss: 0.002558
Iter 300/2000 | Int Loss: 11.1450 | Bndry Loss: 0.000968
Iter 400/2000 | Int Loss: 10.3930 | Bndry Loss: 0.000862
Iter 500/2000 | Int Loss: 10.9503 | Bndry Loss: 0.000777
Iter 600/2000 | Int Loss: 12.8758 | Bndry Loss: 0.000716
Iter 700/2000 | Int Loss: 11.2879 | Bndry Loss: 0.000675
Iter 800/2000 | Int Loss: 11.4573 | Bndry Loss: 0.000615
Iter 900/2000 | Int Loss: 11.8763 | Bndry Loss: 0.000605
Iter 1000/2000 | Int Loss: 12.0341 | Bndry Loss: 0.000540
Iter 1100/2000 | Int Loss: 10.4625 | Bndry Loss: 0.000530
Iter 1200/2000 | Int Loss: 11.3316 | Bndry Loss: 0.000489
Iter 1300/2000 | Int Loss: 11.2336 | Bndry Loss: 0.000479
Iter 1400/2000 | Int Loss: 11.5940 | Bndry Loss: 0.000431
Iter 1500/2000 | Int Loss: 11.3199 | Bndry Loss: 0.000405
Iter 1600/2000 | Int Loss: 11.6795 | Bndry Loss: 0.000413
Iter 1700/2000 | Int Loss: 11.9807 | Bndry Loss: 0.000400
Iter 1800/2000 | Int Lo